In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
import os, argparse
from mne_bids import BIDSPath, read_raw_bids
from TUHProcessingJob import _process_subject
import pandas as pd
import dask.dataframe as dd
import numpy as np
import mne

In [4]:
bids_path = BIDSPath(root="~/mycontainer/TUH_BIDS_Converted", subject="aaaaahhh", session="s0012008", processing="02tcple", recording="t000", task="rest", datatype="eeg", suffix="eeg", extension=".edf")
TUH_patient = read_raw_bids(bids_path, verbose=True)

print(TUH_patient.ch_names)
print(len(TUH_patient.ch_names))

subfolders = [f.path for f in os.scandir("/home/azureuser/mycontainer/TUH_BIDS_Converted") if f.is_dir() ]
subjects = [x.split("/")[-1][4:] for x in subfolders]
print(f"Found {len(subjects)} subjects")

Extracting EDF parameters from /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0012008/eeg/sub-aaaaahhh_ses-s0012008_task-rest_proc-02tcple_rec-t000_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading channel info from /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0012008/eeg/sub-aaaaahhh_ses-s0012008_task-rest_proc-02tcple_rec-t000_channels.tsv.
Reading electrode coords from /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0012008/eeg/sub-aaaaahhh_ses-s0012008_space-CapTrak_electrodes.tsv.
Not fully anonymizing info - keeping his_id, sex, and hand info
['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'A1', 'A2', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']
21
Found 14987 subjects


In [27]:
parser = argparse.ArgumentParser(description='Preprocess TUH EEG data')
parser.add_argument('--input_directory', type=str, help='Path to the directory containing the TUH EEG data', default="/home/azureuser/mycontainer/TUH_BIDS_Converted")
parser.add_argument('--output_dir', type=str, help='Path to the directory where the preprocessed data will be stored', default="./")

args, unknownargs = parser.parse_known_args()

In [28]:
print(args.input_directory, args.output_dir)

/home/azureuser/mycontainer/TUH_BIDS_Converted ./


In [55]:
result = _process_subject(args, "aaaaahhh")
print(result)

Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 23 events and 15000 original time points ...
0 bad epochs dropped
Dropped 2 epochs: 0, 22
None


In [52]:
TUH_patient = TUH_patient.load_data()
TUH_patient.apply_function(lambda x: x * 10000)
print(TUH_patient.get_data().mean(), TUH_patient.get_data().std())

-4228.821932585866 8540.711619659185


[autoreload of preprocessingPipeline failed: Traceback (most recent call last):
  File "/anaconda/envs/EEG-ML/lib/python3.8/site-packages/IPython/extensions/autoreload.py", line 273, in check
    superreload(m, reload, self.old_objects)
  File "/anaconda/envs/EEG-ML/lib/python3.8/site-packages/IPython/extensions/autoreload.py", line 471, in superreload
    module = reload(module)
  File "/anaconda/envs/EEG-ML/lib/python3.8/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 604, in _exec
  File "<frozen importlib._bootstrap_external>", line 843, in exec_module
  File "<frozen importlib._bootstrap>", line 219, in _call_with_frames_removed
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/mchen439-e8ads-v5/code/Users/mchen439/EEG-Foundation-Model/Preprocessing/preprocessingPipeline.py", line 3, in <module>
    from autoreject import AutoReject, RejectLog
ModuleNotFoundError: No module named 'autoreject'
]


In [60]:
from tqdm import tqdm
base = "/home/azureuser/mycontainer/TUH-Processed"
valid_subjects = 0
total_epochs = 0
for subject in tqdm(os.listdir(base)):
	subfolders = [f.path for f in os.scandir(base) if f.is_dir()]
	for subject_folders in subfolders:
		if "epochs_v2" in subject_folders:
			valid_subjects += 1
			for processed_file in os.listdir(os.path.join(base, subject, "epochs_v2")):
				processed_file_path = os.path.join(base, subject, "epochs_v2", processed_file)
				epochs = mne.read_epochs(processed_file_path)
				total_epochs += epochs.get_data().shape[0]

  0%|▎                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            | 5/14991 [00:18<15:18:38,  3.68s/it]


KeyboardInterrupt: 

In [58]:
print(valid_subjects, total_epochs)

0 0
